# Set-Up
Package and data importing

In [ ]:
import warnings
import pandas as pd

from series_analysis import (
    run_stationarity_tests,
    summarize_stationarity,
    plot_stationarity_heatmap,
    run_ljung_box_tests,
    summarize_ljung_box,
    plot_ljung_box_heatmap,
    first_difference,
)
from data_analysis import plot_acf_grid
from visualization import plot_lexical_metrics

warnings.filterwarnings("ignore")

In [ ]:
# Path to the combined CSV written by combine_lexical_csvs.py.
# Edit to match your local or cluster path before running.
CSV_PATH = "/scratch/network/nv9344/Thesis/Thesis-Data/ArcticShift/lexical_df_combined.csv"

# raw_text is excluded via usecols to cut down read time
COLS_TO_LOAD = [
    "utterance_id",
    "speaker_id",
    "subreddit",
    "timestamp",
    "year_month",
    "mtld_score",
    "mattr_score",
    "yules_k",
    "zipf_score",
    "aoa_score",
    "nawl_ratio",
]

METRICS = ["mtld_score", "mattr_score", "yules_k", "zipf_score", "aoa_score", "nawl_ratio"]

df = pd.read_csv(CSV_PATH, usecols=COLS_TO_LOAD, low_memory=False)

# --- Preprocessing ---
# Convert Unix-second timestamp to datetime for time-series plotting.
# (year_month is already pre-computed in the ArcticShift pipeline output.)
df["timestamp"] = pd.to_datetime(df["timestamp"], unit="s", errors="coerce")

# Coerce metric columns to numeric
for m in METRICS:
    df[m] = pd.to_numeric(df[m], errors="coerce")

# Drop rows where fewer than 2 of the 6 metrics are valid
valid_count = df[METRICS].notna().sum(axis=1)
df = df[valid_count >= 2].reset_index(drop=True)

print(f"Loaded {len(df):,} utterances across {df['subreddit'].nunique()} subreddit(s).")
print(df.groupby("subreddit")["year_month"].nunique().rename("n_months").to_frame().T)

# Temporal Analysis
Examine the trends of each metric—all positively correlated with lexical quality—over time

## Initial Visualization

In [ ]:
plot_lexical_metrics(df, resample_freq='ME', agg='mean')

In [ ]:
plot_lexical_metrics(df, resample_freq='Y', agg='mean')

## Aggregation
Aggregate to the subreddit-month level

In [ ]:
# aggregate and sort
agg = df.groupby(["subreddit", "year_month"])[METRICS].mean().reset_index()
agg = agg.sort_values(["subreddit", "year_month"])

In [ ]:
# filter subreddits with insufficient time series length
# ArcticShift spans ~2005-2023 (~216 months max), so 80 obs is a conservative floor
min_obs = 80  # based on DeJong study
counts = agg.groupby("subreddit")["year_month"].count()
valid_subreddits = counts[counts >= min_obs].index
agg = agg[agg["subreddit"].isin(valid_subreddits)]

print(f"Subreddits passing min_obs={min_obs} filter: {sorted(valid_subreddits.tolist())}")
print(counts[counts >= min_obs].rename("n_months").to_frame())

## Pre-Regression Analysis
Confirm the series is stationary and not autocorrelated, adjusting as necessary

### Stationarity Tests
ADF and KPSS tests

In [ ]:
stationarity_results = run_stationarity_tests(agg)
summarize_stationarity(stationarity_results)
plot_stationarity_heatmap(stationarity_results)

### Autocorrelation Tests
Determine if the series is autocorrelated

In [ ]:
plot_acf_grid(agg, n_lags=24)

In [ ]:
lb_results = run_ljung_box_tests(agg)
summarize_ljung_box(lb_results)
plot_ljung_box_heatmap(lb_results)

### First-Differencing

In [ ]:
agg_diff = first_difference(agg)
stationarity_results_diff = run_stationarity_tests(agg_diff)   # should now show stationary
summarize_stationarity(stationarity_results_diff)
plot_stationarity_heatmap(stationarity_results_diff)